In [1]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline, PruneGraph
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
from circuit_tracer.subgraph.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [2]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [3]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [4]:
explainer = shap.Explainer(model, tokenizer)

In [49]:
prompts = ['Cat is to kitten as dog is to']

In [50]:
shap_values = explainer(prompts)

In [51]:
shap.plots.text(shap_values)

In [52]:
print(shap_values)

.values =
array([[[-4.22446732e-06],
        [ 6.54286429e+00],
        [ 1.68508452e+00],
        [ 1.20230146e+00],
        [ 6.04736818e+00],
        [ 6.77137808e-02],
        [ 4.40673760e+00],
        [ 4.12085123e-01],
        [ 4.40843961e-01]]])

.base_values =
array([[-21.73993724]])

.data =
(array(['', 'Cat', ' is', ' to', ' kitten', ' as', ' dog', ' is', ' to'],
      dtype=object),)


In [83]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00082431 0.57229986 0.00444546 0.00274313 0.34868432 0.00088207
 0.06759516 0.00124469 0.001281  ]


## Generate graph if not exist

In [74]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="catistokittenas2",
    source_set = 'gemmascope-transcoder-16k',
    out_path="temp_graph_files/puppy_plt.json",
    node_threshold = 0.95,
    edge_threshold = 0.98,
)

requesting graph for slug='catistokittenas2' prompt='Cat is to kitten as dog is to...'


downloading graph json from https://neuronpedia-attrib.s3.us-east-1.amazonaws.com/user-graphs/anonymous/catistokittenas2.json
saved catistokittenas2 -> temp_graph_files/puppy_plt.json (nodes=4820)


## ASG Pipeline

### Prune

In [84]:
json_path = "temp_graph_files/puppy_plt.json"
# json_path = "temp_graph_files/future-qwen.json"
# source_set = 'gemmascope-transcoder-16k' #'clt-hp' # gemmascope-transcoder-16k
# token_weights = [0.00198786, 0.03153391, 0.00083086, 0.01473883, 0.22338926, 0.00649094,
#  0.00222269, 0.01996207, 0.0052309, 0.67869559, 0.01491708]
# token_weights = [0, 1/3, 0, 0, 1/3, 0, 1/3, 0, 0]
# token_weights = [0, 0, 1/4, 0, 0, 1/4, 0, 1/4, 1/4, 0, 0]
# token_weights = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1/3,0,0,1/3,0,1/3,0,0,0,0,0,0,0,0,0,0,0]
prune_graph = prune_graph_pipeline(
    json_path=json_path,
    logit_weights='target',
    token_weights=token_weights,
    node_influence_threshold=0.6,
    edge_influence_threshold=0.9,
    node_relevance_threshold=0.8,
    edge_relevance_threshold=0.9,
    keep_all_tokens_and_logits=False,
)

In [85]:
print(len(prune_graph.kept_ids))

30


In [87]:
from circuit_tracer.subgraph.utils import get_clerp

get_clerp(prune_graph.metadata, prune_graph.attr)

Found existing clerp for 1_11076_4:  words related to veterinary medicine, especially ...
Found existing clerp for 4_11720_4:  the word "cat" including related terms...
Found existing clerp for 7_11190_6: words related to dogs and associated actions in a ...
Found existing clerp for 8_1093_6:  the virus Porcine Reproductive and Respiratory Sy...
Found existing clerp for 14_13415_4:  words related to animals, particularly dogs, incl...
Found existing clerp for 14_13415_6:  words related to animals, particularly dogs, incl...
Found existing clerp for 16_3708_6: dog...
Found existing clerp for 16_3708_8: dog...
Found existing clerp for 18_595_6:  words related to dogs and dog parks...
Found existing clerp for 18_1686_6: references to dogs and children together...
Found existing clerp for 18_595_8:  words related to dogs and dog parks...
Found existing clerp for 19_4630_8:  phrases including "as a child"...
Found existing clerp for 19_7920_8: segments related to dog behavior...
Found exist

In [88]:
for i, node_id in enumerate(prune_graph.kept_ids):
    print(node_id, prune_graph.attr[node_id].get("clerp", ""), prune_graph.node_influence[i].item(), prune_graph.node_relevance[i].item())

1_11076_4  words related to veterinary medicine, especially diseases in farm animals 0.537501871585846 0.7895568013191223
4_11720_4  the word "cat" including related terms 0.5417401790618896 0.7658317685127258
7_11190_6 words related to dogs and associated actions in a fairly sexual context. 0.49491170048713684 0.7883498668670654
8_1093_6  the virus Porcine Reproductive and Respiratory Syndrome (PRRSV) and related immunology terms. 0.5816071033477783 0.7807872891426086
14_13415_4  words related to animals, particularly dogs, including their emotions and health 0.5662703514099121 0.6994884610176086
14_13415_6  words related to animals, particularly dogs, including their emotions and health 0.48927563428878784 0.7149860858917236
16_3708_6 dog 0.5673236846923828 0.7257762551307678
16_3708_8 dog 0.56191486120224 0.7207479476928711
18_595_6  words related to dogs and dog parks 0.5194127559661865 0.6020478010177612
18_1686_6 references to dogs and children together 0.5153667330741882 0.71790

In [15]:
from pathlib import Path
import torch

def save_prune_snapshot(
    out_path: str,
    prune_graph: PruneGraph,
):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "kept_ids": prune_graph.kept_ids,
        "pruned_adj": prune_graph.pruned_adj.detach().cpu() if hasattr(prune_graph.pruned_adj, "detach") else prune_graph.pruned_adj,
        "node_inf": prune_graph.node_influence.detach().cpu() if hasattr(prune_graph.node_influence, "detach") else prune_graph.node_influence,
        "node_rel": prune_graph.node_relevance.detach().cpu() if hasattr(prune_graph.node_relevance, "detach") else prune_graph.node_relevance,
        "attr": prune_graph.attr,
        "metadata": prune_graph.metadata,
    }
    torch.save(payload, out_path)

def load_prune_snapshot(path: str):
    return torch.load(path, map_location="cpu")

In [89]:
save_prune_snapshot('subgraph/puppy_plt_shap_06_08.pt', prune_graph)

In [43]:
load_prune_snapshot('subgraph/eye_clt_shap_07_075.pt')

{'kept_ids': ['0_51434_10',
  '0_92029_10',
  '16_982_10',
  '17_89134_10',
  '18_16039_10',
  '18_65930_10',
  '19_24530_10',
  '20_66128_10',
  '20_85340_10',
  '25_288_10',
  'E_47641_2',
  'E_575_9',
  'E_573_10',
  '27_6312_10'],
 'pruned_adj': tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  6.0625,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000, -0.9844,  4.4375,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.6055,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  2.5625,  0.0000,  0.0000,  0.0000],
         [ 0.0000,  0.0000,  0.0000,  0.7656,  0.0000,  0.0000,  0.0000,  0.0000,
           0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
         [ 0.000

### Classify features

Heuristic classify alg is terrible, needs improvement or just use LLM

In [ ]:
# feature_types = classify_features(kept_ids, attr, metadata)
# print(feature_types)

IndexError: list index out of range

In [11]:
node_ids = [node_id for node_id in kept_ids if attr[node_id].get("feature_type", "") == "cross layer transcoder"]
# remove 21_61721_10, 22_74186_10
node_ids = [node_id for node_id in node_ids if node_id not in ["21_61721_10", "22_74186_10"]]
print(node_ids)

['16_89970_9', '18_45367_10', '18_56027_10', '19_8271_10', '19_62362_10', '20_44686_9', '20_14775_10', '20_44686_10', '20_74108_10', '21_16875_10', '21_84338_10', '22_11998_10', '22_26199_10', '22_32893_10', '22_43081_10', '22_81571_10', '23_29602_10', '23_31673_10', '23_59746_10', '24_79427_10']


In [8]:
labels = {node_id: attr[node_id].get("clerp", "") for node_id in node_ids }
print(labels)

{'16_89970_9': 'Texas', '18_45367_10': 'US cities', '18_56027_10': 'Dallas', '19_8271_10': 'Foreign words', '19_62362_10': 'city names', '20_44686_9': 'Texas locations/legal contexts', '20_14775_10': 'XML code', '20_44686_10': 'Texas locations/legal contexts', '20_74108_10': 'capital', '21_16875_10': 'cities', '21_84338_10': 'Cities and locations', '22_11998_10': 'Texas and Dallas', '22_26199_10': 'Texas', '22_32893_10': 'Texas legal documents', '22_43081_10': 'technical contexts', '22_81571_10': 'Texas locations and organizations', '23_29602_10': 'City or region names', '23_31673_10': 'Court appeals at specific location', '23_59746_10': 'Technical/Scientific writing', '24_79427_10': ' Kara'}


In [9]:
feature_types = {
    node_id: "Abstract" for node_id in node_ids
}
feature_types['1_89326_9'] = "Input"
print(feature_types)

{'16_89970_9': 'Abstract', '18_45367_10': 'Abstract', '18_56027_10': 'Abstract', '19_8271_10': 'Abstract', '19_62362_10': 'Abstract', '20_44686_9': 'Abstract', '20_14775_10': 'Abstract', '20_44686_10': 'Abstract', '20_74108_10': 'Abstract', '21_16875_10': 'Abstract', '21_84338_10': 'Abstract', '22_11998_10': 'Abstract', '22_26199_10': 'Abstract', '22_32893_10': 'Abstract', '22_43081_10': 'Abstract', '22_81571_10': 'Abstract', '23_29602_10': 'Abstract', '23_31673_10': 'Abstract', '23_59746_10': 'Abstract', '24_79427_10': 'Abstract', '1_89326_9': 'Input'}


In [12]:
supernodes = grouping_pipeline(
    node_ids = node_ids,
    labels = labels,
    feature_types = feature_types,
    prompt = prompts[0],
    model_name = 'gemini-2.5-flash',
)
print(supernodes)

[['Texas State Context', '16_89970_9', '22_11998_10', '22_26199_10'], ['General Geographic Entities', '18_45367_10', '19_62362_10', '21_16875_10', '21_84338_10', '23_29602_10'], ['Texas Legal/Specific Contexts', '20_44686_9', '20_44686_10', '22_32893_10', '22_81571_10', '23_31673_10'], ['Miscellaneous/Irrelevant', '19_8271_10', '20_14775_10', '22_43081_10', '23_59746_10', '24_79427_10']]


### Visualize on the web

In [ ]:
import json
from circuit_tracer.subgraph.api import save_subgraph

# Expected format for save_subgraph:
# supernodes = [
#   ["label_1", "node_id_a", "node_id_b"],
#   ["label_2", "node_id_c", "node_id_d", "node_id_e"],
# ]

status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug="austin",                       # parent graph slug
    displayName="austin-grouped-v1",     # subgraph name shown on Neuronpedia
    pinnedIds=PruneGraph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.7,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality